# Notebook 11 — Decision Trees: Cluster Structure and Outlier Detection

## Purpose

This notebook trains shallow decision trees to (1) recover the cluster structure from the internal clustering variables, and (2) identify the profile of outlier patients. It serves both as an interpretability tool for the HDBSCAN output and as a first step toward a triage-time prediction rule.

---

## Cluster configuration
Two clustering solutions are analysed: **5 clusters** (mcs=3407, ms=170) and **9 clusters** (mcs=2271, ms=15), loaded from their respective labelled CSV files.

---

## Pipeline Overview

### Tree 1 — Cluster structure (internal variables, outliers excluded)
A `DecisionTreeClassifier` (max depth 8, min leaf 300) is trained on the 16 Scenario 2 clustering variables (imaging modalities, biology flags, EKG, disposition counts) with the cluster label as target. Outputs:
- Tree diagram exported as PDF and PNG
- Feature importance bar chart (Gini)
- Full text rule listing

### Tree 2 — Outlier detection (binary target, full dataset)
A second tree (max depth 8, min leaf 40, balanced class weights) is trained to predict `is_outlier` (1 for HDBSCAN label = −1). Rule paths leading to outlier-majority leaves are extracted and printed, characterising the combination of resource use patterns that define the outlier population.

### Trees 3–4 — External features (triage and admission variables)
Additional trees are trained on ordinal-encoded triage variables (`bp_status_ordinal`, `hr_status_ordinal`, etc.), demographic variables (`age`, `sex_bin`, `transport_ordinal`), and optionally `triage` level, both with and without triage score, to assess how well admission information alone predicts cluster membership.

# I. FULL DATASET / 5 clusters (mcs 1900)

## 1. Decision tree for internal variables

In [54]:
# ── 0. Config ──────────────────────────────────────────────────────────────────

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree, _tree
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import dtreeviz
import os

plt.style.use('default')

# ── Paths ─────────────────────────────────────────────────────────────────────
FULL_OUTPUT_DIR = "Results/Regular_clustering/Full_dataset"

run_label = "s2_balanced"
scaler    = "minmax"

FINAL_PARAMS = {
    #"s2_balanced_minmax": (3407, 170),
    "s2_balanced_minmax" : (2271, 15),
}

mcs, ms = FINAL_PARAMS[f"{run_label}_{scaler}"]

CSV_PATH = os.path.join(
    FULL_OUTPUT_DIR, "With_counts",
    scaler, run_label,
    f"final_mcs{mcs}_ms{ms}",
    f"clustering_{run_label}_{scaler}_mcs{mcs}_ms{ms}_labeled.csv"
)

OUT_DIR = os.path.join(
    FULL_OUTPUT_DIR, "With_counts",
    scaler, run_label,
    f"final_mcs{mcs}_ms{ms}",
    "decision_tree"
)



os.makedirs(OUT_DIR, exist_ok=True)
print(f"Input : {CSV_PATH}")
print(f"Output: {OUT_DIR}")

# ── Load dataset ───────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH, low_memory=False)

print(f"Dataset loaded : {len(df)} rows, {df.shape[1]} columns")

# ── 1. Feature definition ──────────────────────────────────────────────────────

IMAGING_COLS_BOOL = {
    "has_ultrasound", "has_ct_scan", "has_xray",
    "has_mri",
    #"has_radio_interventional", "has_nuclear_medicine",
}

BIO_EXAMS = {
    "has_blood_test", "has_culture",
    "has_lumbar_puncture", "has_blood_gas",
}

PROCEDURE_COLS   = {"had_ekg"}

DISPOSITION_COLS = {
    "hospitalization", "observation_unit",
    #"inter_facility_transfer",
}

COLS_QUANTI  = ["imaging_exam_count", "bio_exam_count"]
COLS_BOOL    = list(IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | DISPOSITION_COLS)
ALL_FEATURES = COLS_BOOL + COLS_QUANTI

#Ordre logique du plus au moins consommateur
#5 clusters
CLUSTER_LABELS_5 = {
    1:  "C1 — UHCD + hospitalization + heavy workup",
    0:  "C2 — Hospitalized + full workup",
    2:  "C3 — Discharged + biology +/- ECG",
    4:  "C4 — Discharged + isolated X-ray +/- CT",
    3:  "C5 — Discharged + minimal consumption",
    -1: "Outliers",
}
CLUSTER_ORDER_5 = ["C1 — UHCD + hospitalization + heavy workup",
                 "C2 — Hospitalized + full workup",
                 "C3 — Discharged + biology +/- ECG",
                 "C4 — Discharged + isolated X-ray +/- CT",
                 "C5 — Discharged + minimal consumption",
                 "Outliers"]


# 9 clusters
CLUSTER_LABELS_9 = {
            0 : "C1 — UHCD + hospitalization + mixed workup ++",
            8 : "C2 — Hospitalized + bio +++ (bloodtest++, culture+++, bloodgas+) + imaging+(ctscan) + ECG+",
            5 : "C3 — Hospitalized + blood test++ CTScan++ ECG+",
            6 : "C4 — Hospitalized + blood test++ MRI+++ ECG++",
            4 : "C5 — Hospitalized + blood test + ECG +/- X-ray ++",
            7 : "C6 — Hospitalized + isolated imaging (xray,ctscan)",
            1 : "C7 — Discharged + biology + ECG+",
            3 : "C8 — Discharged + isolated X-ray",
            2 : "C9 — Discharged + minimal consumption",
            -1: "Outliers",
        }

CLUSTER_ORDER_9 = [
            "C1 — UHCD + hospitalization + mixed workup ++",
            "C2 — Hospitalized + bio +++ (bloodtest++, culture+++, bloodgas+) + imaging+(ctscan) + ECG+",
            "C3 — Hospitalized + blood test++ CTScan++ ECG+",
            "C4 — Hospitalized + blood test++ MRI+++ ECG++",
            "C5 — Hospitalized + blood test + ECG +/- X-ray ++",
            "C6 — Hospitalized + isolated imaging (xray,ctscan)",
            "C7 — Discharged + biology + ECG+",
            "C8 — Discharged + isolated X-ray",
            "C9 — Discharged + minimal consumption",
            "Outliers",
        ]


FEATURE_RENAME = {
    'bp_status_ordinal':                  'Blood pressure',
    'hr_status_ordinal':                  'Heart rate',
    'temp_status_ordinal':                'Temperature',
    'sat_status_ordinal':                 'SpO2',
    'rr_status_ordinal':                  'Respiratory rate',
    'o2_flow_status_ordinal':             'O2 flow',
    'gcs_status_ordinal':                 'GCS',
    'cap_blood_sugar_status_ordinal':     'Blood sugar',
    'pupils_status_ordinal':              'Pupils size',
    'anisocoria_status_ordinal':          'Anisocoria',
    'urine_dipstick_clean_status_ordinal':'Urine dipstick',
    'pain_status_ordinal':                'Pain',
    'breathalyzer_status_ordinal':        'Breathalyzer',
    'hemocue_status_ordinal':             'Hemocue',
    'transport_ordinal':                  'Transport mode',
    'age':                                'Age',
    'triage_grouped':                    'Triage level grouped',
    'sex_bin':                             'Sex',
}

STATUS_ORDINAL_FEATURES = [
    'bp_status_ordinal', 'hr_status_ordinal', 'temp_status_ordinal',
    'sat_status_ordinal', 'rr_status_ordinal', 'o2_flow_status_ordinal',
    'gcs_status_ordinal', 'cap_blood_sugar_status_ordinal', 'pupils_status_ordinal',
    'anisocoria_status_ordinal', 'urine_dipstick_clean_status_ordinal',
    'pain_status_ordinal', 'breathalyzer_status_ordinal',
    'hemocue_status_ordinal',
]

EXTERNAL_FEATURES = (
    ["age","triage_grouped","transport_ordinal", "sex_bin"]
    + STATUS_ORDINAL_FEATURES
)

EXTERNAL_FEATURES_NO_TRIAGE = [
    "age", "transport_ordinal", "sex_bin"
] + STATUS_ORDINAL_FEATURES
# ── 2. Prepare datasets ────────────────────────────────────────────────────────

df_model = df[ALL_FEATURES + ['cluster', 'cluster_label']].dropna()

# Tree 1 : clusters only (outliers removed)
df_clusters = df_model[df_model['cluster'] != -1]
X_clusters  = df_clusters[ALL_FEATURES]
y_clusters  = df_clusters['cluster_label']

# Tree 2 : outlier detection (everyone, binary target)
df_outliers               = df_model.copy()
df_outliers['is_outlier'] = (df_outliers['cluster'] == -1).astype(int)
X_outliers                = df_outliers[ALL_FEATURES]
y_outliers                = df_outliers['is_outlier']

print(f"\n── Tree 1 : cluster structure ──")
print(f"Individuals : {len(df_clusters)}")
print(f"Cluster distribution :\n{y_clusters.value_counts()}")



# ── 3. Helper : export tree image ─────────────────────────────────────────────

def export_tree_image(tree, feature_names, class_names, title, filename):
    fig, ax = plt.subplots(figsize=(32, 14), facecolor='white')
    ax.set_facecolor('white')
    plot_tree(
        tree,
        feature_names=feature_names,
        class_names=class_names,
        filled=True,
        rounded=True,
        fontsize=8,
        ax=ax
    )
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{filename}.pdf", format="pdf", dpi=150,
                bbox_inches='tight', facecolor='white')
    plt.savefig(f"{OUT_DIR}/{filename}.png", format="png", dpi=200,
                bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Exported : {filename}.pdf and {filename}.png")

# ── 4. Helper : export feature importance ─────────────────────────────────────

def export_feature_importance(tree, feature_names, title, filename):
    importances = pd.Series(tree.feature_importances_, index=feature_names)
    importances = importances[importances > 0].sort_values(ascending=False)
    print(f"\nFeature importance ({title}) :")
    print(importances.round(3))
    fig, ax = plt.subplots(figsize=(8, 6), facecolor='white')
    ax.set_facecolor('white')
    importances.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(title)
    ax.set_xlabel("Importance (Gini)")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{filename}.png", dpi=200,
                bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Exported : {filename}.png")

# ── 5. Helper : extract rules leading to a target class ───────────────────────

def get_target_rules(tree, feature_names, target_class):
    tree_      = tree.tree_
    classes    = tree.classes_
    feat_names = [
        feature_names[i] if i != _tree.TREE_UNDEFINED else "undefined"
        for i in tree_.feature
    ]
    rules = []

    def recurse(node, conditions):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name      = feat_names[node]
            threshold = tree_.threshold[node]
            recurse(tree_.children_left[node],  conditions + [f"{name} <= {threshold:.2f}"])
            recurse(tree_.children_right[node], conditions + [f"{name} >  {threshold:.2f}"])
        else:
            predicted_class = classes[np.argmax(tree_.value[node])]
            n_samples       = int(tree_.n_node_samples[node])
            purity          = float(np.max(tree_.value[node]) / n_samples)
            if predicted_class == target_class:
                rules.append({
                    'conditions': conditions,
                    'n_samples' : n_samples,
                    'purity'    : purity,
                })

    recurse(0, [])
    return rules



Input : Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs2271_ms15/clustering_s2_balanced_minmax_mcs2271_ms15_labeled.csv
Output: Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs2271_ms15/decision_tree
Dataset loaded : 56784 rows, 194 columns

── Tree 1 : cluster structure ──
Individuals : 52089
Cluster distribution :
cluster_label
C9 — Discharged + minimal consumption                                                         12509
C1 — UHCD + hospitalization + mixed workup ++                                                 10306
C8 — Discharged + isolated X-ray                                                               5995
C5 — Hospitalized + blood test + ECG +/- X-ray ++                                              5422
C3 — Hospitalized + blood test++ CTScan++ ECG+                                                 5379
C7 — Discharged + biology + ECG+                                                               4286
C6 

In [55]:
# ══════════════════════════════════════════════════════════════════════════════
# TREE 1 — Cluster structure (outliers excluded) INTERNAL
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("TREE 1 — Cluster structure")
print("="*60)

tree_clusters = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=300,
    random_state=42
)
tree_clusters.fit(X_clusters, y_clusters)

class_names_clusters = [f"Cluster {c}" for c in sorted(y_clusters.unique())]

export_tree_image(
    tree_clusters,
    ALL_FEATURES,
    class_names_clusters,
    title    = "Decision tree — Cluster structure (Scenario 2)",
    filename = "tree1_clusters"
)

export_feature_importance(
    tree_clusters,
    ALL_FEATURES,
    title    = "Most discriminating features — Cluster structure",
    filename = "tree1_feature_importance"
)

rules_clusters = export_text(tree_clusters, feature_names=ALL_FEATURES)
print("\nText rules :")
print(rules_clusters)

# ══════════════════════════════════════════════════════════════════════════════
# TREE 2 — Outlier detection (full dataset, binary target)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("TREE 2 — Outlier detection")
print("="*60)

tree_outliers = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=40,
    random_state=42,
    class_weight='balanced'
)
tree_outliers.fit(X_outliers, y_outliers)

export_tree_image(
    tree_outliers,
    ALL_FEATURES,
    class_names = ["Non-outlier", "Outlier"],
    title       = "Decision tree — Outlier detection (Scenario 2)",
    filename    = "tree2_outliers"
)

export_feature_importance(
    tree_outliers,
    ALL_FEATURES,
    title    = "Most discriminating features — Outlier detection",
    filename = "tree2_feature_importance"
)

rules_outliers = export_text(tree_outliers, feature_names=ALL_FEATURES)
print("\nText rules :")
print(rules_outliers)

# ── Outlier paths ──────────────────────────────────────────────────────────────

outlier_rules = get_target_rules(tree_outliers, ALL_FEATURES, target_class=1)

print("\n── Paths leading to outlier leaves ──")
if not outlier_rules:
    print("No leaf predicts outliers as majority class.")
    print("Try reducing max_depth or min_samples_leaf.")
else:
    for r in outlier_rules:
        print(f"\n  n={r['n_samples']} individuals (purity {r['purity']:.0%})")
        for cond in r['conditions']:
            print(f"    {cond}")


TREE 1 — Cluster structure
Exported : tree1_clusters.pdf and tree1_clusters.png

Feature importance (Most discriminating features — Cluster structure) :
observation_unit      0.219
hospitalization       0.205
imaging_exam_count    0.167
has_blood_test        0.156
has_ct_scan           0.091
has_mri               0.064
bio_exam_count        0.059
has_culture           0.038
had_ekg               0.000
has_xray              0.000
dtype: float64
Exported : tree1_feature_importance.png

Text rules :
|--- observation_unit <= 0.50
|   |--- has_blood_test <= 0.50
|   |   |--- imaging_exam_count <= 0.50
|   |   |   |--- hospitalization <= 0.50
|   |   |   |   |--- class: C9 — Discharged + minimal consumption
|   |   |   |--- hospitalization >  0.50
|   |   |   |   |--- class: C6 — Hospitalized + isolated imaging (xray,ctscan)
|   |   |--- imaging_exam_count >  0.50
|   |   |   |--- hospitalization <= 0.50
|   |   |   |   |--- imaging_exam_count <= 1.50
|   |   |   |   |   |--- class: C8 — Di

# decision tree for external variables

In [56]:
#=============================
# building tree feature
#==============================

## encoding with all modalities
# STATUS_ENCODINGS = {
#
#     # ── Pression artérielle ───────────────────────────────────────────────────
#     'bp_status': {
#         'not_measured' : 1,
#         'hypotension'  : 2,
#         'normotension' : 3,
#         'hypertension' : 4,
#     },
#
#     # ── Fréquence cardiaque ───────────────────────────────────────────────────
#     'hr_status': {
#         'not_measured' : 1,
#         'bradycardia'  : 2,
#         'normocardia'  : 3,
#         'tachycardia'  : 4,
#     },
#
#     # ── Température ──────────────────────────────────────────────────────────
#     'temp_status': {
#         'not_measured' : 1,
#         'hypothermia'  : 2,
#         'normothermia' : 3,
#         'hyperthermia' : 4,
#     },
#
#     # ── Saturation O2 ─────────────────────────────────────────────────────────
#     'sat_status': {
#         'not_measured' : 1,
#         'severe hypoxia': 2,
#         'hypoxia'      : 3,
#         'normal'       : 4,
#     },
#
#     # ── Fréquence respiratoire ────────────────────────────────────────────────
#     'rr_status': {
#         'not_measured' : 1,
#         'bradypnea'    : 2,
#         'normal'       : 3,
#         'tachypnea'    : 4,
#     },
#
#     # ── O2 flow — binaire ─────────────────────────────────────────────────────
#     'o2_flow_status': {
#         'not_measured' : 0,
#         'off'          : 1,
#         'on'           : 2,
#     },
#
#     # ── Glasgow ───────────────────────────────────────────────────────────────
#     'gcs_status': {
#         'not_measured'      : 1,
#         'severe_impairment' : 2,
#         'moderate_impairment': 3,
#         'normal'            : 4,
#     },
#
#     # ── Glycémie capillaire ───────────────────────────────────────────────────
#     'cap_blood_sugar_status': {
#         'not_measured' : 1,
#         'hypoglycemia' : 2,
#         'normoglycemia': 3,
#         'hyperglycemia': 4,
#     },
#
#     # ── Pupilles ──────────────────────────────────────────────────────────────
#     'pupils_status': {
#         'not_measured' : 1,
#         'myosis'       : 2,
#         'normal'       : 3,
#         'mydriasis'    : 4,
#     },
#
#     # ── Anisocorie — binaire ──────────────────────────────────────────────────
#     'anisocoria_status': {
#         'not_measured' : 0,
#         'no'           : 1,
#         'yes'          : 2,
#     },
#
#     # ── Bandelette urinaire — binaire ─────────────────────────────────────────
#     'urine_dipstick_clean_status': {
#         'not_measured' : 0,
#         'negative'     : 1,
#         'positive'     : 2,
#     },
#
#     # ── Douleur ───────────────────────────────────────────────────────────────
#     'pain_status': {
#         'not_measured' : 1,
#         'no_pain'      : 2,
#         'mild_pain'    : 3,
#         'moderate_pain': 4,
#         'severe_pain'  : 5,
#     },
#
#     # ── Ethylotest — binaire ──────────────────────────────────────────────────
#     'breathalyzer_status': {
#         'not_measured' : 0,
#         'negative'     : 1,
#         'positive'     : 2,
#     },
#
#     # ── Hémocue ───────────────────────────────────────────────────────────────
#     'hemocue_status': {
#         'not_measured'  : 1,
#         'severe_anemia' : 2,
#         'moderate_anemia': 3,
#         'normal'        : 4,
#     },
# }


## encoding only with not measured, normal and abnormal
STATUS_ENCODINGS = {

    'bp_status': {
        'not_measured' : 0,
        'hypotension'  : 2,   # abnormal
        'normotension' : 1,   # normal
        'hypertension' : 2,   # abnormal
    },

    'hr_status': {
        'not_measured' : 0,
        'bradycardia'  : 2,
        'normocardia'  : 1,
        'tachycardia'  : 2,
    },

    'temp_status': {
        'not_measured' : 0,
        'hypothermia'  : 2,
        'normothermia' : 1,
        'hyperthermia' : 2,
    },

    'sat_status': {
        'not_measured'  : 0,
        'severe hypoxia': 2,
        'hypoxia'       : 2,
        'normal'        : 1,
    },

    'rr_status': {
        'not_measured' : 0,
        'bradypnea'    : 2,
        'normal'       : 1,
        'tachypnea'    : 2,
    },

    'o2_flow_status': {
        'not_measured' : 0,
        'off'          : 1,   # normal (pas d'O2)
        'on'           : 2,   # abnormal (O2 nécessaire)
    },

    'gcs_status': {
        'not_measured'       : 0,
        'severe_impairment'  : 2,
        'moderate_impairment': 2,
        'normal'             : 1,
    },

    'cap_blood_sugar_status': {
        'not_measured' : 0,
        'hypoglycemia' : 2,
        'normoglycemia': 1,
        'hyperglycemia': 2,
    },

    'pupils_status': {
        'not_measured' : 0,
        'myosis'       : 2,
        'normal'       : 1,
        'mydriasis'    : 2,
    },

    'anisocoria_status': {
        'not_measured' : 0,
        'no'           : 1,
        'yes'          : 2,
    },

    'urine_dipstick_clean_status': {
        'not_measured' : 0,
        'negative'     : 1,
        'positive'     : 2,
    },

    'pain_status': {
        'not_measured' : 0,
        'no_pain'      : 1,
        'mild_pain'    : 2,
        'moderate_pain': 2,
        'severe_pain'  : 2,
    },

    'breathalyzer_status': {
        'not_measured' : 0,
        'negative'     : 1,
        'positive'     : 2,
    },

    'hemocue_status': {
        'not_measured'   : 0,
        'severe_anemia'  : 2,
        'moderate_anemia': 2,
        'normal'         : 1,
    },
}







# ── Application status encodings ──────────────────────────────────────────────
for col, mapping in STATUS_ENCODINGS.items():
    if col in df.columns:
        df[f'{col}_ordinal'] = df[col].map(mapping)
        n_nan = df[f'{col}_ordinal'].isna().sum()
        if n_nan > 0:
            print(f"⚠️  {col}: {n_nan} NaN non mappés")
        else:
            print(f"✅ {col}: encodé ({len(mapping)} modalités)")

# ── Transport ─────────────────────────────────────────────────────────────────
TRANSPORT_ORDER = {
    'Unknown'            : 1,
    'Personal'           : 2,
    'Post medical advice': 3,
    'Ambulance'          : 4,
    'Emergency services' : 5,
}
df['transport_ordinal'] = df['transport_grouped'].map(TRANSPORT_ORDER)
print("Distribution transport_ordinal :")
print(df['transport_ordinal'].value_counts().sort_index())
print(f"NaN restants : {df['transport_ordinal'].isna().sum()}")

# ── Triage groupé : 1+2 → urgent, 3 → semi-urgent, 4+5 → non-urgent ──────────
TRIAGE_GROUP_MAP = {
    1: 1,   # urgent/critical
    2: 1,
    3: 2,   # semi-urgent
    4: 3,   # non-urgent
    5: 3,
}
df['triage_grouped'] = df['triage'].map(TRIAGE_GROUP_MAP)
print("Distribution triage_grouped :")
print(df['triage_grouped'].value_counts().sort_index())
print(f"Remaining NaN : {df['triage_grouped'].isna().sum()}")

# ── Encoding function ─────────────────────────────────────────────────────────────────
def encode_features(df):
    for col, mapping in STATUS_ENCODINGS.items():
        ordinal_col = f'{col}_ordinal'
        if ordinal_col not in df.columns:
            if col in df.columns:
                df[ordinal_col] = df[col].map(mapping)
    if 'transport_ordinal' not in df.columns:
        df['transport_ordinal'] = df['transport_grouped'].map(TRANSPORT_ORDER)
    # Triage groupé
    if 'triage_grouped' not in df.columns:
        TRIAGE_GROUP_MAP = {1: 1, 2: 1, 3: 2, 4: 3, 5: 3}
        df['triage_grouped'] = df['triage'].map(TRIAGE_GROUP_MAP)
    return df

✅ bp_status: encodé (4 modalités)
✅ hr_status: encodé (4 modalités)
✅ temp_status: encodé (4 modalités)
✅ sat_status: encodé (4 modalités)
✅ rr_status: encodé (4 modalités)
✅ o2_flow_status: encodé (3 modalités)
✅ gcs_status: encodé (4 modalités)
✅ cap_blood_sugar_status: encodé (4 modalités)
✅ pupils_status: encodé (4 modalités)
✅ anisocoria_status: encodé (3 modalités)
✅ urine_dipstick_clean_status: encodé (3 modalités)
✅ pain_status: encodé (5 modalités)
✅ breathalyzer_status: encodé (3 modalités)
✅ hemocue_status: encodé (4 modalités)
Distribution transport_ordinal :
transport_ordinal
1     2397
2    31547
3      305
4     7489
5    15046
Name: count, dtype: int64
NaN restants : 0
Distribution triage_grouped :
triage_grouped
1    15979
2    20857
3    19948
Name: count, dtype: int64
Remaining NaN : 0


In [57]:
    # ============================================================
    # MODULAR CALLS — EXTERNAL DECISION TREES FOR 5 & 9 CLUSTERS
    # ============================================================


    CLUSTERING_RUNS = [
        {
            "cluster_solution": 5,
            "run_label": "s2_balanced",
            "scaler": "minmax",
            "mcs": 3407,
            "ms": 170,
            "cluster_labels": CLUSTER_LABELS_5,
            "cluster_order": CLUSTER_ORDER_5,
        },
        {
            "cluster_solution": 9,
            "run_label": "s2_balanced",
            "scaler": "minmax",
            "mcs": 2271,
            "ms": 15,
            "cluster_labels": CLUSTER_LABELS_9,
            "cluster_order": CLUSTER_ORDER_9,
        },
    ]


    for config in CLUSTERING_RUNS:

        # ========================================================
        # CONFIG
        # ========================================================

        cluster_solution = config["cluster_solution"]
        run_label        = config["run_label"]
        scaler           = config["scaler"]
        mcs              = config["mcs"]
        ms               = config["ms"]
        CLUSTER_LABELS   = config["cluster_labels"]
        CLUSTER_ORDER    = config["cluster_order"]

        print("\n" + "=" * 80)
        print(f"RUNNING DECISION TREES — {cluster_solution} CLUSTERS")
        print("=" * 80)

        # ========================================================
        # PATHS
        # ========================================================
        CSV_PATH = os.path.join(
            FULL_OUTPUT_DIR, "With_counts",
            scaler, run_label,
            f"final_mcs{mcs}_ms{ms}",
            f"clustering_{run_label}_{scaler}_mcs{mcs}_ms{ms}_labeled.csv"
        )

        OUT_DIR_TREE = os.path.join(
            FULL_OUTPUT_DIR,
            "With_counts",
            scaler,
            run_label,
            f"final_mcs{mcs}_ms{ms}",
            f"decision_tree_external_{cluster_solution}clusters"
        )

        os.makedirs(OUT_DIR_TREE, exist_ok=True)

        print(f"Input : {CSV_PATH}")
        print(f"Output: {OUT_DIR_TREE}")

        # ========================================================
        # LOAD DATA
        # ========================================================

        df = pd.read_csv(CSV_PATH, low_memory=False)
        df = encode_features(df)

        # ========================================================
        # ORDINAL LEGEND EXPORT
        # ========================================================

    #### legend for reeal features with all modalities

        # fig, ax = plt.subplots(figsize=(10, 12), facecolor="white")
        # ax.axis("off")
        #
        # table_data = [["Variable", "Value", "Label"]]
        #
        # for col, mapping in STATUS_ENCODINGS.items():
        #     col_clean = FEATURE_RENAME.get(
        #         f"{col}_ordinal",
        #         col.replace("_status", "").replace("_", " ")
        #     )
        #
        #     for label, value in sorted(mapping.items(), key=lambda x: x[1]):
        #         table_data.append([
        #             col_clean,
        #             str(value),
        #             label.replace("_", " ")
        #         ])
        #
        # table = ax.table(
        #     cellText=table_data[1:],
        #     colLabels=table_data[0],
        #     cellLoc="left",
        #     loc="center",
        #     colWidths=[0.35, 0.1, 0.35],
        # )
        #
        # table.auto_set_font_size(False)
        # table.set_fontsize(9)
        # table.scale(1, 1.4)
        #
        # for j in range(3):
        #     table[0, j].set_facecolor("#2c3e50")
        #     table[0, j].set_text_props(color="white", fontweight="bold")
        #
        # prev_var = None
        # color_a = "#f5f5f5"
        # color_b = "#ffffff"
        # current_color = color_a
        #
        # for i in range(1, len(table_data)):
        #     var = table_data[i][0]
        #     if var != prev_var:
        #         current_color = color_b if current_color == color_a else color_a
        #         prev_var = var
        #     for j in range(3):
        #         table[i, j].set_facecolor(current_color)
        #
        # ax.set_title(
        #     f"Ordinal encoding legend — {cluster_solution}-cluster solution",
        #     fontsize=12,
        #     fontweight="bold",
        #     pad=20
        # )
        #
        # plt.tight_layout()
        # plt.savefig(
        #     os.path.join(OUT_DIR_TREE, "ordinal_legend.png"),
        #     dpi=200,
        #     bbox_inches="tight",
        #     facecolor="white"
        # )
        # plt.close()


        ### Legend for feature not measured normal abnormal


        LEGEND_DATA = {
            **{
                FEATURE_RENAME.get(f'{col}_ordinal', col): {
                    0: 'not measured',
                    1: 'normal',
                    2: 'abnormal',
                }
                for col in STATUS_ENCODINGS.keys()
            },
            'Triage level (grouped)': {
                1: 'urgent/critical (triage 1+2)',
                2: 'semi-urgent (triage 3)',
                3: 'non-urgent (triage 4+5)',
            },
            'Transport mode': {
                1: 'Unknown',
                2: 'Personal',
                3: 'Post medical advice',
                4: 'Ambulance',
                5: 'Emergency services',
            },
        }

        table_data = [["Variable", "Value", "Label"]]
        for var_name, mapping in LEGEND_DATA.items():
            for value, label in sorted(mapping.items()):
                table_data.append([var_name, str(value), label])

        fig, ax = plt.subplots(figsize=(10, len(table_data) * 0.35 + 1), facecolor="white")
        ax.axis("off")

        table = ax.table(
            cellText=table_data[1:],
            colLabels=table_data[0],
            cellLoc="left",
            loc="center",
            colWidths=[0.40, 0.10, 0.40],
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 1.4)

        for j in range(3):
            table[0, j].set_facecolor("#2c3e50")
            table[0, j].set_text_props(color="white", fontweight="bold")

        prev_var = None
        color_a, color_b = "#f5f5f5", "#ffffff"
        current_color = color_a
        for i in range(1, len(table_data)):
            var = table_data[i][0]
            if var != prev_var:
                current_color = color_b if current_color == color_a else color_a
                prev_var = var
            for j in range(3):
                table[i, j].set_facecolor(current_color)

        ax.set_title(
            f"Ordinal encoding legend — {cluster_solution}-cluster solution",
            fontsize=12, fontweight="bold", pad=20
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(OUT_DIR_TREE, "ordinal_legend.png"),
            dpi=200, bbox_inches="tight", facecolor="white"
        )
        plt.close()
        print("Legend exported → ordinal_legend.png")

        # ========================================================
        # PREPARE DATA
        # ========================================================

        cols_needed = EXTERNAL_FEATURES + ["cluster_label"]
        df_model = df[[c for c in cols_needed if c in df.columns]].dropna()

        # ========================================================
        # INTERNAL TREES — clustering features (ALL_FEATURES)
        # ========================================================

        OUT_DIR_INTERNAL = os.path.join(
            FULL_OUTPUT_DIR, "With_counts",
            scaler, run_label,
            f"final_mcs{mcs}_ms{ms}",
            f"decision_tree_internal_{cluster_solution}clusters"
        )
        os.makedirs(OUT_DIR_INTERNAL, exist_ok=True)

        df_model_internal = df[ALL_FEATURES + ['cluster', 'cluster_label']].dropna()

        df_clust_internal = df_model_internal[df_model_internal['cluster_label'] != "Outliers"].copy()
        X_int = df_clust_internal[ALL_FEATURES]
        y_int = df_clust_internal['cluster_label']

        tree_int = DecisionTreeClassifier(
            max_depth=8,
            min_samples_leaf=300,
            random_state=42
        )
        tree_int.fit(X_int, y_int)

        # Feature importance
        imp_int = pd.Series(tree_int.feature_importances_, index=ALL_FEATURES)
        imp_int = imp_int[imp_int > 0].sort_values(ascending=False)

        fig, ax = plt.subplots(figsize=(9, max(5, len(imp_int) * 0.4)), facecolor="white")
        imp_int.sort_values().plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
        ax.set_title(f"Feature Importance — Internal Tree ({cluster_solution} clusters)")
        ax.set_xlabel("Gini Importance")
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR_INTERNAL, f"internal_tree_importance_{cluster_solution}clusters.png"),
                    dpi=200, bbox_inches="tight", facecolor="white")
        plt.close()

        # Rules
        rules_int = export_text(tree_int, feature_names=ALL_FEATURES)
        with open(os.path.join(OUT_DIR_INTERNAL, f"internal_tree_rules_{cluster_solution}clusters.txt"), "w") as f:
            f.write(rules_int)

        print(f"Internal tree saved → {OUT_DIR_INTERNAL}")



        # ========================================================
        # TREE 1 EXTERNAL — MULTICLASS CLUSTERS
        # ========================================================

        df_clusters = df_model[df_model["cluster_label"] != "Outliers"].copy()

        X1 = df_clusters[EXTERNAL_FEATURES].astype(float)
        X1 = X1.rename(columns=FEATURE_RENAME)


        le1 = LabelEncoder()
        y1 = le1.fit_transform(df_clusters["cluster_label"])
        class_names1 = list(le1.classes_)

        print(f"Tree 1 — N={len(X1)} | Classes: {class_names1}")

        tree1 = DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=200,
            class_weight="balanced",
            random_state=42,
        )

        tree1.fit(X1, y1)

        viz1 = dtreeviz.model(
            tree1,
            X_train=X1,
            y_train=y1,
            feature_names=list(X1.columns),
            class_names=class_names1,
            target_name="Cluster",
        )

        v1 = viz1.view(fancy=True, scale=1.5, orientation="LR")
        v1.save(os.path.join(
            OUT_DIR_TREE,
            f"tree1_clusters_{cluster_solution}clusters.svg"
        ))

        v1_simple = viz1.view(fancy=False, scale=1.2, orientation="TD")
        v1_simple.save(os.path.join(
            OUT_DIR_TREE,
            f"tree1_clusters_{cluster_solution}clusters_simple.svg"
        ))

        # Feature importance
        imp1 = pd.Series(tree1.feature_importances_, index=X1.columns)
        imp1 = imp1[imp1 > 0].sort_values(ascending=False)

        fig, ax = plt.subplots(
            figsize=(9, max(5, len(imp1) * 0.4)),
            facecolor="white"
        )

        imp1.sort_values().plot(
            kind="barh",
            ax=ax,
            color="steelblue",
            edgecolor="white"
        )

        ax.axvline(
            imp1.mean(),
            color="red",
            linestyle="--",
            alpha=0.6,
            label="Mean"
        )

        ax.set_title(
            f"Feature Importance — Tree 1 ({cluster_solution} clusters)",
            fontsize=13,
            fontweight="bold"
        )

        ax.set_xlabel("Gini Importance")
        ax.legend()

        plt.tight_layout()
        plt.savefig(
            os.path.join(
                OUT_DIR_TREE,
                f"tree1_feature_importance_{cluster_solution}clusters.png"
            ),
            dpi=200,
            bbox_inches="tight"
        )
        plt.close()

        # Rules
        rules1 = export_text(tree1, feature_names=list(X1.columns))

        with open(
            os.path.join(
                OUT_DIR_TREE,
                f"tree1_rules_{cluster_solution}clusters.txt"
            ),
            "w"
        ) as f:
            f.write(rules1)

        # ========================================================
        # TREE 2 EXTERNAL — BINARY OUTLIER DETECTION
        # ========================================================

        df_all = df_model.copy()

        X2 = df_all[EXTERNAL_FEATURES].astype(float)
        X2 = X2.rename(columns=FEATURE_RENAME)

        y2 = (df_all["cluster_label"] == "Outliers").astype(int)

        class_names2 = ["Clustered", "Outlier"]

        print(
            f"Tree 2 — N={len(X2)} | "
            f"Outliers: {y2.sum()} ({y2.mean():.1%})"
        )

        tree2 = DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=200,
            class_weight="balanced",
            random_state=42,
        )

        tree2.fit(X2, y2)

        viz2 = dtreeviz.model(
            tree2,
            X_train=X2,
            y_train=y2,
            feature_names=list(X2.columns),
            class_names=class_names2,
            target_name="Outlier",
        )

        v2 = viz2.view(fancy=True, scale=1.5, orientation="LR")
        v2.save(os.path.join(
            OUT_DIR_TREE,
            f"tree2_outliers_{cluster_solution}clusters.svg"
        ))

        v2_simple = viz2.view(fancy=False, scale=1.2, orientation="TD")
        v2_simple.save(os.path.join(
            OUT_DIR_TREE,
            f"tree2_outliers_{cluster_solution}clusters_simple.svg"
        ))

        # Feature importance
        imp2 = pd.Series(tree2.feature_importances_, index=X2.columns)
        imp2 = imp2[imp2 > 0].sort_values(ascending=False)

        fig, ax = plt.subplots(
            figsize=(9, max(5, len(imp2) * 0.4)),
            facecolor="white"
        )

        imp2.sort_values().plot(
            kind="barh",
            ax=ax,
            color="tomato",
            edgecolor="white"
        )

        ax.axvline(
            imp2.mean(),
            color="red",
            linestyle="--",
            alpha=0.6,
            label="Mean"
        )

        ax.set_title(
            f"Feature Importance — Tree 2 ({cluster_solution} clusters outliers)",
            fontsize=13,
            fontweight="bold"
        )

        ax.set_xlabel("Gini Importance")
        ax.legend()

        plt.tight_layout()
        plt.savefig(
            os.path.join(
                OUT_DIR_TREE,
                f"tree2_feature_importance_{cluster_solution}clusters.png"
            ),
            dpi=200,
            bbox_inches="tight"
        )
        plt.close()

        # Rules
        rules2 = export_text(tree2, feature_names=list(X2.columns))

        with open(
            os.path.join(
                OUT_DIR_TREE,
                f"tree2_rules_{cluster_solution}clusters.txt"
            ),
            "w"
        ) as f:
            f.write(rules2)

        print(f"Completed {cluster_solution}-cluster analysis.")


        # ========================================================
        # TREE 1b & 2b — WITHOUT TRIAGE
        # ========================================================

        OUT_DIR_NOTRIAGE = os.path.join(
            FULL_OUTPUT_DIR, "With_counts",
            scaler, run_label,
            f"final_mcs{mcs}_ms{ms}",
            f"decision_tree_external_notriage_{cluster_solution}clusters"
        )
        os.makedirs(OUT_DIR_NOTRIAGE, exist_ok=True)

        cols_needed_nt = EXTERNAL_FEATURES_NO_TRIAGE + ["cluster_label"]
        df_model_nt    = df[[c for c in cols_needed_nt if c in df.columns]].dropna()

        # ── Tree 1b — multiclass clusters sans triage ──────────────────────────────
        df_clusters_nt = df_model_nt[df_model_nt["cluster_label"] != "Outliers"].copy()
        X1b = df_clusters_nt[EXTERNAL_FEATURES_NO_TRIAGE].astype(float)
        X1b = X1b.rename(columns=FEATURE_RENAME)

        le1b = LabelEncoder()
        y1b  = le1b.fit_transform(df_clusters_nt["cluster_label"])
        class_names1b = list(le1b.classes_)

        tree1b = DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=200,
            class_weight="balanced",
            random_state=42,
        )
        tree1b.fit(X1b, y1b)

        viz1b = dtreeviz.model(
            tree1b,
            X_train=X1b,
            y_train=y1b,
            feature_names=list(X1b.columns),
            class_names=class_names1b,
            target_name="Cluster",
        )
        viz1b.view(fancy=True, scale=1.5, orientation="LR").save(
            os.path.join(OUT_DIR_NOTRIAGE, f"tree1b_clusters_notriage_{cluster_solution}clusters.svg")
        )
        viz1b.view(fancy=False, scale=1.2, orientation="TD").save(
            os.path.join(OUT_DIR_NOTRIAGE, f"tree1b_clusters_notriage_{cluster_solution}clusters_simple.svg")
        )

        imp1b = pd.Series(tree1b.feature_importances_, index=X1b.columns)
        imp1b = imp1b[imp1b > 0].sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(9, max(5, len(imp1b) * 0.4)), facecolor="white")
        ax.set_facecolor("white")
        imp1b.sort_values().plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
        ax.axvline(imp1b.mean(), color="red", linestyle="--", alpha=0.6, label="Mean")
        ax.set_title(f"Feature Importance — Tree 1b no triage ({cluster_solution} clusters)", fontsize=13, fontweight="bold")
        ax.set_xlabel("Gini Importance")
        ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR_NOTRIAGE, f"tree1b_feature_importance_notriage_{cluster_solution}clusters.png"),
                    dpi=200, bbox_inches="tight", facecolor="white")
        plt.close()

        with open(os.path.join(OUT_DIR_NOTRIAGE, f"tree1b_rules_notriage_{cluster_solution}clusters.txt"), "w") as f:
            f.write(export_text(tree1b, feature_names=list(X1b.columns)))

        # ── Tree 2b — outlier detection sans triage ────────────────────────────────
        X2b = df_model_nt[EXTERNAL_FEATURES_NO_TRIAGE].astype(float)
        X2b = X2b.rename(columns=FEATURE_RENAME)
        y2b = (df_model_nt["cluster_label"] == "Outliers").astype(int)

        tree2b = DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=200,
            class_weight="balanced",
            random_state=42,
        )
        tree2b.fit(X2b, y2b)

        viz2b = dtreeviz.model(
            tree2b,
            X_train=X2b,
            y_train=y2b,
            feature_names=list(X2b.columns),
            class_names=["Clustered", "Outlier"],
            target_name="Outlier",
        )
        viz2b.view(fancy=True, scale=1.5, orientation="LR").save(
            os.path.join(OUT_DIR_NOTRIAGE, f"tree2b_outliers_notriage_{cluster_solution}clusters.svg")
        )
        viz2b.view(fancy=False, scale=1.2, orientation="TD").save(
            os.path.join(OUT_DIR_NOTRIAGE, f"tree2b_outliers_notriage_{cluster_solution}clusters_simple.svg")
        )

        imp2b = pd.Series(tree2b.feature_importances_, index=X2b.columns)
        imp2b = imp2b[imp2b > 0].sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(9, max(5, len(imp2b) * 0.4)), facecolor="white")
        ax.set_facecolor("white")
        imp2b.sort_values().plot(kind="barh", ax=ax, color="tomato", edgecolor="white")
        ax.axvline(imp2b.mean(), color="red", linestyle="--", alpha=0.6, label="Mean")
        ax.set_title(f"Feature Importance — Tree 2b no triage ({cluster_solution} clusters outliers)", fontsize=13, fontweight="bold")
        ax.set_xlabel("Gini Importance")
        ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR_NOTRIAGE, f"tree2b_feature_importance_notriage_{cluster_solution}clusters.png"),
                    dpi=200, bbox_inches="tight", facecolor="white")
        plt.close()

        with open(os.path.join(OUT_DIR_NOTRIAGE, f"tree2b_rules_notriage_{cluster_solution}clusters.txt"), "w") as f:
            f.write(export_text(tree2b, feature_names=list(X2b.columns)))

        print(f"No-triage trees saved → {OUT_DIR_NOTRIAGE}")


RUNNING DECISION TREES — 5 CLUSTERS
Input : Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs3407_ms170/clustering_s2_balanced_minmax_mcs3407_ms170_labeled.csv
Output: Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs3407_ms170/decision_tree_external_5clusters
Legend exported → ordinal_legend.png


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

Internal tree saved → Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs3407_ms170/decision_tree_internal_5clusters
Tree 1 — N=54886 | Classes: ['C1 — UHCD + hospitalization + heavy workup', 'C2 — Hospitalized + full workup', 'C3 — Discharged + biology +/- ECG', 'C4 — Discharged + isolated X-ray +/- CT', 'C5 — Discharged + minimal consumption']


/home/nadia/pycharm_project_nad/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Fon

Tree 2 — N=56784 | Outliers: 1898 (3.3%)


...

Tree 2 — N=56784 | Outliers: 4695 (8.3%)


/home/nadia/pycharm_project_nad/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Fon

Completed 9-cluster analysis.


/home/nadia/pycharm_project_nad/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Fon

No-triage trees saved → Results/Regular_clustering/Full_dataset/With_counts/minmax/s2_balanced/final_mcs2271_ms15/decision_tree_external_notriage_9clusters
